### **Immune Cell Signatures**

- Sade-Feldman et al.

In [1]:
import pandas as pd

# Define your file path and the name of the sheet
FILE_PATH = 'mmc1.xlsx'
SHEET_NAME = 'Gene marker-Fig1B-C' # Change this to your sheet name
GENE_COLUMN_NAME = 'GeneName'
SF_ImmuneSignatures = {}

In [2]:
g1_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,       # Skips the first 4 rows (loads row 5 as header)
    nrows=228-2,
    usecols='A:D'# Reads the next 10 data rows
)

g1_df.head()

SF_ImmuneSignatures["G1_Bcells"] = g1_df["GeneName"].values.tolist()

In [3]:
g1_df

,GeneName,P-value,Mean expression G1,Mean expression non-G1
0,IGHD,<1e-300,5.377109,0.024127
1,PAX5,<1e-300,4.830784,0.023214
2,FCRL1,<1e-300,5.758715,0.033841
3,CR2,<1e-300,3.177965,0.026061
4,VPREB3,<1e-300,3.779632,0.031216
...,...,...,...,...
221,MKNK2,0.0,2.665456,1.860346
222,RGS19,0.0,2.616485,1.829179
223,KIAA0226,0.0,3.019653,2.113321
224,CD69,0.0,7.453068,5.232637


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
from scipy import stats

source_data_path = "/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline/ExperimentNSCLC/LungCancer_ICB/Source Data/"
source_data_path_clinical = source_data_path + 'Clinical/'
source_data_path_exome = source_data_path + 'Exome/'
source_data_path_rna = source_data_path + 'RNA/'
source_data_path_ref = source_data_path + 'Reference/'
source_data_path_int = source_data_path + 'Integrative/'
source_data_path_out = source_data_path + 'Output/'

rna_df = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_rnaseqc_tpm_v1.gct',skiprows=2,sep='\t')
rna_df = rna_df.drop(columns = ["Name"])
rna_df = rna_df.set_index("Description").T

In [4]:
su2c_is_sf_harm = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_Curated_Sets_SF_v1.txt',sep='\t')
su2c_is_sf_harm.head()

,Harmonized_SU2C_RNA_Tumor_Sample_ID_v2,B-cells,Cytotoxic cells,DC,Exhausted CD8,Exhausted/HS CD8,Lymphocytes,Lymphocytes exhausted/cell cycle,Macrophages/Monocytes,Memory T cells,Plasma,Treg
0,SU2CLC-CLE-NIVO10-T1,-1.259768,-0.842559,-0.433253,-1.422601,-1.774418,-1.350288,-2.821135,-0.297431,-2.075795,-1.973561,-2.756701
1,SU2CLC-CLE-NIVO18-T1,0.710736,0.063517,0.001486,0.039338,-0.141059,0.250194,0.147430,-0.288875,0.147272,1.199513,0.686839
2,SU2CLC-CLE-NIVO19-T1,0.525296,0.338409,0.408307,0.465818,0.409422,-0.066813,0.698741,-0.443927,-0.243725,0.313025,0.444627
3,SU2CLC-CLE-NIVO2-T1,0.768654,0.997207,-0.068318,0.931905,0.860628,1.111493,0.529933,-0.186005,0.663986,0.181901,0.321043
4,SU2CLC-CLE-NIVO20-T1,0.698067,2.250859,-0.619655,2.899739,2.006238,1.680539,2.192263,-0.464790,0.364394,-0.104678,0.926042


In [5]:
su2c_is_zi_harm = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_Curated_Sets_ZI_v1.txt',sep='\t')
su2c_is_zi_harm.head()

,Harmonized_SU2C_RNA_Tumor_Sample_ID_v2,hMø1,hMø4,hMø5,hMø6,hMø7,hMø8,hMø9,hMono1,hMono2,hMono3
0,SU2CLC-CLE-NIVO10-T1,-1.109117,-1.156333,-2.190216,-1.200923,-1.991091,-0.699990,-0.672497,-2.092137,-2.369909,-0.657237
1,SU2CLC-CLE-NIVO18-T1,0.789262,-0.241140,-0.269748,0.133515,-0.600769,-0.509239,-0.120329,-0.341126,0.238928,-0.473816
2,SU2CLC-CLE-NIVO19-T1,0.766106,1.190017,-0.606071,0.155629,-0.065903,-0.128578,1.258769,-0.274179,-0.066995,0.329074
3,SU2CLC-CLE-NIVO2-T1,-0.930557,0.196371,1.801350,-0.306568,-0.883556,2.732102,1.364896,-0.042298,-0.055184,-0.387409
4,SU2CLC-CLE-NIVO20-T1,0.280883,-0.904412,-0.644487,0.228805,-0.496632,-0.017237,1.022745,0.450452,1.109786,-0.185438


In [4]:
su2c_limma_harm = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_Limma_All_v1.txt',sep='\t')
su2c_limma_harm.head()

,Unnamed: 0,logFC,AveExpr,t,P.Value,adj.P.Val,B,ensembl_gene_id_version,hgnc_symbol,median_log2tpm,gexp_gt1_percent,gexp_gt1_cat
0,1,-0.403370,1.093088,-1.635074,0.104315,0.403476,-4.315271,ENSG00000187634.11,SAMD11,1.117063,0.785124,50-80%
1,2,0.088979,5.565879,1.021167,0.308964,0.632204,-5.437019,ENSG00000188976.10,NOC2L,5.311830,0.983471,80-100%
2,3,-0.019629,3.103165,-0.153923,0.877896,0.954957,-5.669993,ENSG00000187961.13,KLHL17,3.266475,0.983471,80-100%
3,4,0.194845,2.217655,0.922449,0.357906,0.669481,-5.190195,ENSG00000187583.10,PLEKHN1,2.603013,0.958678,80-100%
4,5,0.110187,-1.277853,0.368140,0.713334,0.883703,-5.027598,ENSG00000187642.9,PERM1,0.504215,0.512397,50-80%


In [6]:
def tpm_to_log2tpm(tpm,
                   pseudo_count: float = 1.0,
                   fillna_with_zero: bool = True):

    if isinstance(tpm, pd.DataFrame):
        mat = tpm.copy()
        if fillna_with_zero:
            mat = mat.fillna(0.0)
        return np.log2(mat + pseudo_count)
    else:
        arr = np.array(tpm, dtype=float, copy=True)
        if fillna_with_zero:
            # replace NaN with 0
            arr = np.nan_to_num(arr, nan=0.0)
        return np.log2(arr + pseudo_count)

log2tpm_df = tpm_to_log2tpm(rna_df,pseudo_count=1)
log2tpm_df.head()

Description,DDX11L1,WASH7P,MIR6859-1,MIR1302-2HG,MIR1302-2,FAM138A,OR4G4P,OR4G11P,OR4F5,RP11-34P13.7,...,MT-ND4,MT-TH,MT-TS2,MT-TL2,MT-ND5,MT-ND6,MT-TE,MT-CYB,MT-TT,MT-TP
SU2CLC-CLE-NIVO10-T1,0.000000,1.447902,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.009485,...,2.891481,0.0,0.000000,0.0,4.463544,2.143126,0.000000,1.752427,0.0,0.000000
SU2CLC-CLE-NIVO18-T1,0.000000,3.830570,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,4.590913,0.0,0.000000,0.0,3.702580,3.583928,0.000000,3.431420,0.0,0.000000
SU2CLC-CLE-NIVO19-T1,0.551221,4.187934,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.013528,...,5.445952,0.0,1.130858,0.0,4.534852,4.835242,0.000000,4.936671,0.0,0.000000
SU2CLC-CLE-NIVO2-T1,0.733902,3.370739,0.0,0.0,0.0,0.0,0.0,0.0,1.526049,0.000000,...,6.590475,0.0,0.000000,0.0,5.550211,5.980782,1.512333,5.413547,0.0,2.100951
SU2CLC-CLE-NIVO20-T1,0.054428,3.462458,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.050752,...,7.708408,0.0,0.000000,0.0,6.784792,6.591271,0.975811,6.240276,0.0,0.000000


In [10]:
#G3 Monocytes Macrophages
def correlation_signatures(signature, sig_col):
    g3_signatures = signature
    g3_tpm_exp = log2tpm_df[g3_signatures]
    normalized_g3_tpm_exp = zscore(g3_tpm_exp.mean(axis =1, skipna=False))
    # print(sig_col, ':',stats.spearmanr(
    #     normalized_g3_tpm_exp.values,
    #     su2c_is_sf_harm[sig_col].values  
    # ))
    corr, _ = stats.spearmanr(
        normalized_g3_tpm_exp.values,
        su2c_is_sf_harm[sig_col].values  
    )
    return corr, normalized_g3_tpm_exp.values,  su2c_is_sf_harm[sig_col].values

In [11]:
SF_signatures = {}
g1_sig = ['PAX5',"CD19","MS4A1","IGHM","BLNK"]
g1_col = 'B-cells'
SF_signatures[g1_col] = g1_sig

a,b,c = correlation_signatures(g1_sig, g1_col)

In [32]:
SF_signatures = {}
g1_sig = ['PAX5',"CD19","MS4A1","IGHM","BLNK"]
g1_col = 'B-cells'
SF_signatures[g1_col] = g1_sig

g2_sig = ['TNFRSF17',"XBP1","IGHG4","IGHG3","IGHA2"]
g2_col = 'Plasma' 
SF_signatures[g2_col] = g2_sig

g3_sig = ['FCN1','VCAN','CD14','CD33','CSF3R']
g3_col = 'Macrophages/Monocytes'
SF_signatures[g3_col] = g3_sig

g4_sig = ['CLEC4C','LILRA4','SERPINF1','IL3RA','THBD']
g4_col = 'DC'
SF_signatures[g4_col] = g4_sig

g5_sig = ['IL7R','TCF7','GZMK','FYN','CD8A']
g5_col = 'Lymphocytes'
SF_signatures[g5_col] = g5_sig

g6_sig = ['LAG3','FASLG','HAVCR2','PDCD1','CD38','CD8A','CD8B']
g6_col = 'Exhausted CD8'
SF_signatures[g6_col] = g6_sig

g7_sig = ['FOXP3','IL2RA','CD4','CTLA4']
g7_col = 'Treg'
SF_signatures[g7_col] = g7_sig

g8_sig = ['FCGR3A','KLRG1','PRF1','GZMB','CD8A']
g8_col = 'Cytotoxic cells'
SF_signatures[g8_col] = g8_sig

g9_sig = ['CD8A','CD8B','CTLA4','HAVCR2','ENTPD1','HSPH1','HSPB1']
g9_col = 'Exhausted/HS CD8'
SF_signatures[g9_col] = g9_sig

g10_sig = ['TCF7','IL7R','SELL','LTB','LEF1']
g10_col = 'Memory T cells'
SF_signatures[g10_col] = g10_sig

# g11_sig = ['CDCA5','CDC6','TOP2A','MKI67','HAVCR2','PDCD1','LAG3','ENTPD1']
g11_sig = ['CDCA5','CDC6','TOP2A','MKI67','HAVCR2','PDCD1','LAG3','ENTPD1']
g11_col = 'Lymphocytes exhausted/cell cycle'
SF_signatures[g11_col] = g11_sig

In [34]:
print(correlation_signatures(g8_sig, g8_col))

0.9686404363069729


In [35]:
import itertools 
def discover_combinations(gene_signatures, name_signature):
    
    all_combinations = []
    correlations = []
    for r in range(1, len(gene_signatures) + 1):
        for combo in itertools.combinations(gene_signatures, r):
            if len(combo) == 1:
                all_combinations.append([combo[0]])
                corr = correlation_signatures([combo[0]], name_signature)
                correlations.append(corr)
            else:
                all_combinations.append(list(combo))
                corr = correlation_signatures(list(combo), name_signature)
                correlations.append(corr)
        

    max_corr = np.max(correlations)
    max_combination = all_combinations[np.argmax(correlations)]

    print(f"Optimal combination {max_combination} for {name_signature} signature: Correlation {max_corr}")
    return max_combination, max_corr


Optimal_SF_signature = {}
for name,  signature in SF_signatures.items():
    print(f'-----{name}-----')
    g1_combination, max_corr = discover_combinations(signature, name)
    Optimal_SF_signature[name] = [g1_combination, max_corr]

-----B-cells-----
Optimal combination ['PAX5', 'CD19', 'MS4A1', 'IGHM'] for B-cells signature: Correlation 1.0
-----Plasma-----
Optimal combination ['TNFRSF17', 'XBP1', 'IGHG4', 'IGHG3', 'IGHA2'] for Plasma signature: Correlation 0.9896254075000512
-----Macrophages/Monocytes-----
Optimal combination ['FCN1', 'VCAN', 'CD14', 'CD33', 'CSF3R'] for Macrophages/Monocytes signature: Correlation 1.0
-----DC-----
Optimal combination ['CLEC4C', 'SERPINF1', 'IL3RA', 'THBD'] for DC signature: Correlation 1.0
-----Lymphocytes-----
Optimal combination ['IL7R', 'TCF7', 'GZMK', 'CD8A'] for Lymphocytes signature: Correlation 0.9824527231596717
-----Exhausted CD8-----
Optimal combination ['LAG3', 'HAVCR2', 'PDCD1', 'CD8A', 'CD8B'] for Exhausted CD8 signature: Correlation 0.9883166232683382
-----Treg-----
Optimal combination ['FOXP3', 'IL2RA', 'CD4', 'CTLA4'] for Treg signature: Correlation 0.9865157635030309
-----Cytotoxic cells-----
Optimal combination ['FCGR3A', 'KLRG1', 'PRF1', 'GZMB', 'CD8A'] for C

In [51]:
my_sf_sig = {}
# my_sf_sig['Sample'] = log2tpm_df.index.tolist()
for cell, opt_signature in Optimal_SF_signature.items():
    sig_tpm_exp = log2tpm_df[opt_signature[0]]
    normalized_sig_tpm_exp = zscore(sig_tpm_exp.mean(axis =1, skipna=False))
    my_sf_sig[cell] = normalized_sig_tpm_exp

    
my_SF_df = pd.DataFrame(my_sf_sig)   
my_SF_df = my_SF_df[su2c_is_sf_harm.columns.tolist()[1:]]
my_SF_df.to_csv('SU2C-MARK_Harmonized_Curated_Sets_SF_TTC.csv', index = True)

## **MHCI signature**
- Şenbabaoğlu et al

In [15]:
su2c_is_hm_harm = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_Curated_Sets_HM_v1.txt',sep='\t')
su2c_is_hm_harm.head()

,Harmonized_SU2C_RNA_Tumor_Sample_ID_v2,Adenosine (Corvus),Antigen processing machinery (PMID: 27855702),EMT2 (PMID: 27321955),IFNG,Merck/Nanostring 18 gene T cell–inflamed GEP score,NFAT/NR4A1 family T cell dysfunction,TGF-B (Mariathasan Nature 2018)
0,SU2CLC-CLE-NIVO10-T1,-0.347220,-1.781804,-1.511646,-2.566604,-1.651118,-1.074415,-1.277529
1,SU2CLC-CLE-NIVO18-T1,-0.408656,-0.178822,-0.665314,0.322697,-0.179835,-0.578566,-0.553323
2,SU2CLC-CLE-NIVO19-T1,0.670662,0.791995,0.439311,1.339746,0.789172,0.343941,0.595709
3,SU2CLC-CLE-NIVO2-T1,-0.675524,1.059913,0.732698,0.712259,1.031519,1.181614,0.169688
4,SU2CLC-CLE-NIVO20-T1,0.590495,0.876095,-1.018308,0.845423,2.206946,0.182293,-1.137497


In [53]:
mhc_sign = ['HLA-A','HLA-B','HLA-C','B2M','TAP1','TAP2','TAPBP']
mhc_col = 'Antigen processing machinery (PMID: 27855702)'

def correlation_signatures(signature, sig_col, ref_table):
    g3_signatures = signature
    g3_tpm_exp = log2tpm_df[g3_signatures]
    normalized_g3_tpm_exp = zscore(g3_tpm_exp.mean(axis =1, skipna=False))
    # print(sig_col, ':',stats.spearmanr(
    #     normalized_g3_tpm_exp.values,
    #     su2c_is_sf_harm[sig_col].values  
    # ))
    corr, _ = stats.spearmanr(
        normalized_g3_tpm_exp.values,
        ref_table[sig_col].values  
    )
    return corr
    
def discover_combinations(gene_signatures, name_signature, ref_table):
    
    all_combinations = []
    correlations = []
    for r in range(1, len(gene_signatures) + 1):
        for combo in itertools.combinations(gene_signatures, r):
            if len(combo) == 1:
                all_combinations.append([combo[0]])
                corr = correlation_signatures([combo[0]], name_signature, ref_table)
                correlations.append(corr)
            else:
                all_combinations.append(list(combo))
                corr = correlation_signatures(list(combo), name_signature,ref_table)
                correlations.append(corr)
        

    max_corr = np.max(correlations)
    max_combination = all_combinations[np.argmax(correlations)]

    print(f"Optimal combination {max_combination} for {name_signature} signature: Correlation {max_corr}")
    return max_combination, max_corr


optimal_mhc_sig, mhc_corr = discover_combinations(
    mhc_sign, mhc_col, su2c_is_hm_harm
)


Optimal combination ['HLA-A', 'HLA-C', 'B2M', 'TAP1', 'TAP2'] for Antigen processing machinery (PMID: 27855702) signature: Correlation 0.9965725572208667


Adenosine (Corvus) Willingham et al.

In [76]:
a2ar_sig = ['CD8A','CXCL9','CXCL10','EOMES','IFNG','GZMA','GZMB','TBX21','CD274','LAG3','TIGIT'
           ,'CD40','ADK','ADM','ADORA2B','ADORA3','NT5C1A']
a2ar_col = 'Adenosine (Corvus)'
optimal_a2ar_sig, a2ar_corr = discover_combinations(
    a2ar_sig, a2ar_col, su2c_is_hm_harm
)

Optimal combination ['CD8A', 'GZMB', 'CD40', 'ADM', 'ADORA2B', 'ADORA3'] for Adenosine (Corvus) signature: Correlation 0.6150465763161311


In [81]:
su2c_is_hm_harm

,Harmonized_SU2C_RNA_Tumor_Sample_ID_v2,Adenosine (Corvus),Antigen processing machinery (PMID: 27855702),EMT2 (PMID: 27321955),IFNG,Merck/Nanostring 18 gene T cell–inflamed GEP score,NFAT/NR4A1 family T cell dysfunction,TGF-B (Mariathasan Nature 2018)
0,SU2CLC-CLE-NIVO10-T1,-0.347220,-1.781804,-1.511646,-2.566604,-1.651118,-1.074415,-1.277529
1,SU2CLC-CLE-NIVO18-T1,-0.408656,-0.178822,-0.665314,0.322697,-0.179835,-0.578566,-0.553323
2,SU2CLC-CLE-NIVO19-T1,0.670662,0.791995,0.439311,1.339746,0.789172,0.343941,0.595709
3,SU2CLC-CLE-NIVO2-T1,-0.675524,1.059913,0.732698,0.712259,1.031519,1.181614,0.169688
4,SU2CLC-CLE-NIVO20-T1,0.590495,0.876095,-1.018308,0.845423,2.206946,0.182293,-1.137497
...,...,...,...,...,...,...,...,...
147,SU2CLC-UCD-1137-T1,0.093170,-0.300072,0.350756,0.469760,0.908205,-0.050089,0.459129
148,SU2CLC-UCD-1142-T1,1.429329,0.673808,0.266181,0.858295,1.548541,-0.872834,0.521754
149,SU2CLC-UCD-1143-T1,1.410430,-0.215873,1.256896,0.435449,0.580626,2.386251,0.797464
150,SU2CLC-UCD-1557-T1,-0.700387,-0.260763,0.173573,-0.227131,-0.939560,0.883515,-0.652068


## **T-cell inflammed 18 genes**
- Ayers et al

In [83]:
su2c_is_hm_harm.columns.tolist()

['Harmonized_SU2C_RNA_Tumor_Sample_ID_v2',
 'Adenosine (Corvus)',
 'Antigen processing machinery (PMID: 27855702)',
 'EMT2 (PMID: 27321955)',
 'IFNG',
 'Merck/Nanostring 18 gene\xa0T cell–inflamed GEP score',
 'NFAT/NR4A1 family T cell dysfunction',
 'TGF-B (Mariathasan Nature 2018)']

In [85]:
ayers_sig = ['CXCR6','TIGIT','CD27','CD274','PDCD1LG2','LAG3','NKG7','PSMB10','CMKLR1','CD8A','IDO1'
           ,'CCL5','CXCL9','HLA-DQA1','CD276','HLA-DRB1','STAT1','HLA-E']
ayers_col =  'Merck/Nanostring 18 gene\xa0T cell–inflamed GEP score'

optimal_ayers_sig, ayers_corr = discover_combinations(
    ayers_sig, ayers_col, su2c_is_hm_harm
)

Optimal combination ['CXCR6', 'TIGIT', 'CD27', 'LAG3', 'NKG7', 'PSMB10', 'CD8A', 'IDO1', 'CCL5', 'CXCL9', 'STAT1'] for Merck/Nanostring 18 gene T cell–inflamed GEP score signature: Correlation 0.9952398526507151


EMT Transcription Factors (EMT TFs)

In [20]:
import itertools 
def correlation_signatures(signature, sig_col, ref_table):
    g3_signatures = signature
    g3_tpm_exp = log2tpm_df[g3_signatures]
    normalized_g3_tpm_exp = zscore(g3_tpm_exp.mean(axis =1, skipna=False))
    # print(sig_col, ':',stats.spearmanr(
    #     normalized_g3_tpm_exp.values,
    #     su2c_is_sf_harm[sig_col].values  
    # ))
    corr, _ = stats.spearmanr(
        normalized_g3_tpm_exp.values,
        ref_table[sig_col].values  
    )
    return corr
    
def discover_combinations(gene_signatures, name_signature, ref_table):
    
    all_combinations = []
    correlations = []
    for r in range(1, len(gene_signatures) + 1):
        for combo in itertools.combinations(gene_signatures, r):
            if len(combo) == 1:
                all_combinations.append([combo[0]])
                corr = correlation_signatures([combo[0]], name_signature, ref_table)
                correlations.append(corr)
            else:
                all_combinations.append(list(combo))
                corr = correlation_signatures(list(combo), name_signature,ref_table)
                correlations.append(corr)
        

    max_corr = np.max(correlations)
    max_combination = all_combinations[np.argmax(correlations)]

    print(f"Optimal combination {max_combination} for {name_signature} signature: Correlation {max_corr}")
    return max_combination, max_corr

In [21]:
emt2_sig = ['SOX9' ,'TWIST1','FOXF1' , 'ZEB1', 'ZEB2','GATA6']
emt2_col =  'EMT2 (PMID: 27321955)'

optimal_emt_sig, emt_corr = discover_combinations(
    emt2_sig, emt2_col, su2c_is_hm_harm
)

Optimal combination ['SOX9', 'TWIST1', 'FOXF1', 'ZEB1', 'ZEB2', 'GATA6'] for EMT2 (PMID: 27321955) signature: Correlation 0.9900662251655628


NFAT/NR4A1 family T cell dysfunction

In [24]:
nr4a1_sig =  [
    "NR4A1", "NR4A2", "NR4A3",             
    "TOX", "TOX2",             
    "IRF4", "JUN"              
]
nr4a1_col =  'NFAT/NR4A1 family T cell dysfunction'

optimal_nr4a1_sig, nr4a1_corr = discover_combinations(
    nr4a1_sig, nr4a1_col, su2c_is_hm_harm
)

Optimal combination ['NR4A1', 'NR4A2', 'NR4A3'] for NFAT/NR4A1 family T cell dysfunction signature: Correlation 0.9542643129053643


In [31]:
nr4a1_sig =  [
    "NR4A1", "NR4A2", "NR4A3", "NFATC2",             
    "TOX", "TOX2",             
    "IRF4", "JUN"              
]
nr4a1_col =  'NFAT/NR4A1 family T cell dysfunction'

optimal_nr4a1_sig, nr4a1_corr = discover_combinations(
    nr4a1_sig, nr4a1_col, su2c_is_hm_harm
)

Optimal combination ['NR4A1', 'NR4A2', 'NR4A3'] for NFAT/NR4A1 family T cell dysfunction signature: Correlation 0.9542643129053643


TGF-B (Mariathasan Nature 2018)

In [34]:
tgf_sig = [
    "ACTA2", "ACTG2", "ADAM12", "ADAM19", "CNN1", 
    "COL4A1", "CTGF", "CTPS1", "FSTL3", 
    "HSPB1", "IGFBP3", "PXDC1", "SEMA7A", "SH3PXD2A", 
    "TAGLN", "TGFBI", "TNS1", "TPM1"
]
tgf_col =  'TGF-B (Mariathasan Nature 2018)'

optimal_tgf_sig, tgf_corr = discover_combinations(
    tgf_sig, tgf_col, su2c_is_hm_harm
)

Optimal combination ['ACTA2', 'ADAM19', 'CNN1', 'COL4A1', 'SEMA7A', 'TAGLN', 'TGFBI', 'TNS1'] for TGF-B (Mariathasan Nature 2018) signature: Correlation 0.9448089448396995


In [40]:
import numpy as np
import itertools
from functools import lru_cache
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import multiprocessing as mp

def discover_combinations_v3_greedy(gene_signatures, name_signature, ref_table, 
                                     max_size=None, top_k=10):
    """
    Greedy search - don't test all combinations
    
    Strategy:
    1. Test all single signatures
    2. Iteratively add the signature that improves correlation most
    3. Stop when no improvement
    
    Parameters
    ----------
    max_size : int, optional
        Maximum combination size to test
    top_k : int
        Keep top k combinations at each step
    """
    if max_size is None:
        max_size = len(gene_signatures)
    
    # Step 1: Test all single signatures
    print("Step 1: Testing single signatures...")
    single_results = []
    for sig in gene_signatures:
        corr = correlation_signatures([sig], name_signature, ref_table)
        single_results.append(([sig], corr))
    
    # Sort by correlation
    single_results.sort(key=lambda x: x[1], reverse=True)
    
    print(f"  Best single:  {single_results[0][0]} (corr={single_results[0][1]:.4f})")
    
    # Keep track of best overall
    best_combination = single_results[0][0]
    best_corr = single_results[0][1]
    
    # Step 2: Iteratively grow combinations
    current_candidates = single_results[: top_k]  # Keep top k
    
    for size in range(2, max_size + 1):
        print(f"\nStep {size}: Testing combinations of size {size}...")
        
        new_candidates = []
        
        for combo, prev_corr in current_candidates: 
            # Try adding each remaining signature
            remaining = [s for s in gene_signatures if s not in combo]
            
            for sig in remaining:
                new_combo = combo + [sig]
                new_corr = correlation_signatures(new_combo, name_signature, ref_table)
                new_candidates.append((new_combo, new_corr))
        
        if not new_candidates:
            break
        
        # Sort and keep top k
        new_candidates.sort(key=lambda x: x[1], reverse=True)
        current_candidates = new_candidates[:top_k]
        
        # Update best if improved
        if current_candidates[0][1] > best_corr:
            best_combination = current_candidates[0][0]
            best_corr = current_candidates[0][1]
            print(f"  New best:  {best_combination} (corr={best_corr:.4f})")
        else:
            print(f"  No improvement.  Stopping.")
            break
    
    return best_combination, best_corr

In [42]:
tgf_sig = [
    "ACTA2", "ACTG2", "ADAM12", "ADAM19", "CNN1", 
    "COL4A1", "CTGF", "CTPS1", "FSTL3", 
    "HSPB1", "IGFBP3", "PXDC1", "SEMA7A", "SH3PXD2A", 
    "TAGLN", "TGFBI", "TNS1", "TPM1", 'TGFBR2', 
]
tgf_col =  'TGF-B (Mariathasan Nature 2018)'

optimal_tgf_sig, tgf_corr = discover_combinations_v3_greedy(
    tgf_sig, tgf_col, su2c_is_hm_harm
)

Step 1: Testing single signatures...
  Best single:  ['TAGLN'] (corr=0.8871)

Step 2: Testing combinations of size 2...
  New best:  ['TAGLN', 'TGFBR2'] (corr=0.9623)

Step 3: Testing combinations of size 3...
  New best:  ['TAGLN', 'TGFBR2', 'COL4A1'] (corr=0.9756)

Step 4: Testing combinations of size 4...
  No improvement.  Stopping.


In [44]:
def discover_combinations_v3_greedy_patience(gene_signatures, name_signature, ref_table, 
                                              max_size=None, top_k=10, patience=5):
    """
    Greedy search with patience - continue searching even without immediate improvement
    
    Strategy:
    1. Test all single signatures
    2. Iteratively add the signature that improves correlation most
    3. Continue for 'patience' steps without improvement before stopping
    
    Parameters
    ----------
    max_size : int, optional
        Maximum combination size to test (default: all signatures)
    top_k : int
        Keep top k combinations at each step (default: 10)
    patience : int
        Number of steps without improvement before stopping (default: 5)
        Example: If patience=5, will try sizes 6,7,8,9,10 even if 5 didn't improve
    """
    if max_size is None:
        max_size = len(gene_signatures)
    
    # Step 1: Test all single signatures
    print("Step 1: Testing single signatures...")
    single_results = []
    for sig in gene_signatures:
        corr = correlation_signatures([sig], name_signature, ref_table)
        single_results.append(([sig], corr))
    
    # Sort by correlation
    single_results.sort(key=lambda x: x[1], reverse=True)
    
    print(f"  Best single:  {single_results[0][0]} (corr={single_results[0][1]:.4f})")
    
    # Keep track of best overall
    best_combination = single_results[0][0]
    best_corr = single_results[0][1]
    best_size = 1
    
    # Track steps without improvement
    steps_without_improvement = 0
    
    # Step 2: Iteratively grow combinations
    current_candidates = single_results[: top_k]  # Keep top k
    
    for size in range(2, max_size + 1):
        print(f"\nStep {size}: Testing combinations of size {size}...")
        
        new_candidates = []
        
        for combo, prev_corr in current_candidates: 
            # Try adding each remaining signature
            remaining = [s for s in gene_signatures if s not in combo]
            
            for sig in remaining:
                new_combo = combo + [sig]
                new_corr = correlation_signatures(new_combo, name_signature, ref_table)
                new_candidates.append((new_combo, new_corr))
        
        if not new_candidates: 
            print(f"  No new candidates. Stopping.")
            break
        
        # Sort and keep top k
        new_candidates.sort(key=lambda x: x[1], reverse=True)
        current_candidates = new_candidates[:top_k]
        
        # Check if improved
        current_best_corr = current_candidates[0][1]
        
        if current_best_corr > best_corr:
            # Improvement found! 
            best_combination = current_candidates[0][0]
            best_corr = current_best_corr
            best_size = size
            steps_without_improvement = 0  # Reset counter
            print(f"  ✅ NEW BEST:  {best_combination} (corr={best_corr:.4f})")
        else:
            # No improvement
            steps_without_improvement += 1
            print(f"  No improvement (patience:  {steps_without_improvement}/{patience})")
            
            # Check if patience exhausted
            if steps_without_improvement >= patience:
                print(f"  🛑 Stopping:  No improvement for {patience} consecutive steps")
                break
    
    print(f"\n{'='*60}")
    print(f"FINAL RESULT:")
    print(f"Best combination: {best_combination}")
    print(f"Best correlation: {best_corr:.4f}")
    print(f"Combination size: {best_size}")
    print(f"{'='*60}")
    
    return best_combination, best_corr

In [45]:
def tsv_to_python_list(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            # Đọc toàn bộ nội dung file
            content = f.read()
            
            # 1. Thay thế dấu tab bằng dấu phẩy (nếu có)
            # 2. Tách chuỗi thành list dựa trên dấu phẩy
            # 3. Dùng strip() để loại bỏ khoảng trắng hoặc ký tự xuống dòng dư thừa
            raw_list = [gene.strip() for gene in content.replace('\t', ',').split(',') if gene.strip()]
            
            return raw_list
    except FileNotFoundError:
        return "Không tìm thấy file. Vui lòng kiểm tra lại đường dẫn."

# --- SỬ DỤNG ---
file_name = 'HALLMARK_INTERFERON_GAMMA_RESPONSE.v7.5.1.tsv' # Thay bằng tên file thực tế của bạn
genes = tsv_to_python_list(file_name)
genes_ifng = list(set(genes)&set(log2tpm_df.columns.tolist()))

ifng_sig = genes_ifng
ifng_col =  'IFNG'

optimal_ifng_sig, ifng_corr = discover_combinations_v3_greedy_patience(
    ifng_sig, ifng_col, su2c_is_hm_harm
)

Step 1: Testing single signatures...
  Best single:  ['EPSTI1'] (corr=0.8053)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:  ['UBE2L6', 'STAT4'] (corr=0.9052)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:  ['OAS2', 'IRF1', 'NMI'] (corr=0.9399)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:  ['UBE2L6', 'STAT4', 'OAS2', 'IRF1'] (corr=0.9542)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:  ['UBE2L6', 'STAT4', 'OAS2', 'IRF1', 'STAT2'] (corr=0.9642)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:  ['UBE2L6', 'STAT4', 'OAS2', 'IRF1', 'STAT2', 'PSME1'] (corr=0.9681)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:  ['UBE2L6', 'STAT4', 'IFI35', 'IFIH1', 'NFKB1', 'IRF1', 'OAS2'] (corr=0.9737)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:  ['UBE2L6', 'STAT4', 'IFI35', 'IFIH1', 'NFKB1', 'IRF1', 'OAS2', 'ST8SIA4'] (corr=0.9755)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:  ['UBE2L6', 'STAT4', 'IFI35', 'IFIH1', 'NFKB1

ValueError: Format specifier missing precision